# 04 · Protocolo de evaluación y E1
Lógica en `src/protocolo.py` (protocolo) y, más adelante, en el módulo de E1.

**Protocolo rolling-origin con reentrenamiento mensual.** Para cada mes *m* de 2024 (validación):
1. se entrena con las filas cuya etiqueta ya se conoce al empezar *m* (`target_month <= m - latencia`; con latencia 1, hasta el mes anterior);
2. se predicen todas las celdas de *m* a la vez;
3. se pasa al mes siguiente y se vuelve a entrenar con un mes más.

**2025 (test) no se toca.** Todo modelo (E0, E1…) pasa por este mismo protocolo para compararlo en igualdad.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

from baselines import indice_estacional, predecir_e0
from features import anadir_rezagos
from modelos_conteo import ENTRENO_DESDE, contraste_sobredispersion, glm_poisson
from protocolo import (
    MESES_VALIDACION,
    evaluar_rolling,
    resumir,
    resumir_por_banda,
    resumir_por_mes,
)

PROCESSED = ROOT / "data" / "processed"
MESES_VALIDACION

## Paneles
Los rezagos se calculan dentro de `predecir_e0` sobre el panel completo: cada fila del mes *m* solo usa meses anteriores (comprobado en los tests), así que no hay fuga aunque el panel incluya 2025.

In [ ]:
paneles = {n: pd.read_parquet(PROCESSED / f"panel_{n}.parquet") for n in ["1km", "2km", "distritos"]}
paneles["1km"].head()

## E0 por el protocolo
Los baselines no entrenan, pero se pasan por el mismo protocolo que usarán E1–E5. Los cinco de las directrices deben dar **lo mismo** que en el notebook 03.

El sexto, **`media_12_estacional`**, es la media de 12 meses multiplicada por el índice estacional del mes (ver más abajo). Es el **baseline a batir**.

In [ ]:
def evaluar_baselines(panel):
    preds = predecir_e0(panel, MESES_VALIDACION)
    tabla = pd.DataFrame({n: resumir(p) for n, p in preds.items()}).T
    return preds, tabla.round(3)

e0 = {n: evaluar_baselines(p) for n, p in paneles.items()}
e0["1km"][1]

In [ ]:
e0["2km"][1]

In [ ]:
e0["distritos"][1]

## Efecto agosto: índice estacional
Total de cada mes del año dividido entre la media mensual de su año, promediado sobre los años **completos ya conocidos** y **sin 2020** (COVID). Es lo que se sabe al empezar 2024: años 2019, 2021, 2022 y 2023. 2025 no interviene.

Agosto tiene alrededor de un tercio menos de accidentes todos los años (vacaciones); octubre–diciembre, algo más. La media de 12 meses predice casi lo mismo cada mes y por eso falla en agosto.

In [ ]:
indice = indice_estacional(paneles["1km"], pd.Timestamp("2023-12-01"))
indice.round(3).plot.bar(figsize=(7, 3), title="Índice estacional (1 = mes medio)")

## Errores por banda de frecuencia (1 km)
Banda según `ma_12` de la celda (media de los 12 meses anteriores): **0** si vale 0, **1–2** si está entre 0 y 3, **≥3** si es 3 o más. Nunca se usa el `y` del propio mes.

La banda 0 es trivial (todos predicen 0); en **≥3** es donde se distinguen los modelos.

In [ ]:
preds_1km = e0["1km"][0]
pd.concat(
    {n: resumir_por_banda(p).set_index("banda") for n, p in preds_1km.items()}
).round(3)

## Error mes a mes (1 km)
Media de 12 meses frente a su versión estacional: la corrección se nota sobre todo en agosto.

In [ ]:
por_mes = pd.DataFrame({
    n: resumir_por_mes(preds_1km[n]).set_index("target_month")["MAE"]
    for n in ["media_movil_12", "media_12_estacional"]
}).round(3)
por_mes

In [ ]:
por_mes.plot(marker="o", figsize=(7, 3), title="MAE por mes en 2024 (1 km)")

## E1 · GLM Poisson (L2) con offset y mes del año
Código en `src/modelos_conteo.py`. Modelo **multiplicativo**:

`predicción = (ma_12 + 0,01) · exp(b0 + efecto del mes)`, es decir, `log(μ) = log(ma_12 + c) + b0 + 11 indicadoras de mes`.

- El **offset** `log(ma_12 + c)` hace la predicción proporcional al nivel histórico de la celda.
- Las **11 indicadoras de mes** captan el efecto agosto sin que se lo digamos a mano.
- Se reentrena cada mes por el protocolo. **Solo se entrena con filas desde 2022**: 2020 está distorsionado por la COVID y las filas de 2021 tienen meses de 2020 en su ventana de 12 meses.
- Una primera versión con `log1p(historial)` quedó por debajo de la media móvil (MAE 1,04): no era proporcional al historial y una celda sin accidentes recibía ~0,23. Se descartó. Añadir `lag_1`, `lag_12`, `ma_3` y `ma_6` tampoco mejoró (una vez conocida `ma_12` y el mes, el historial reciente no aporta).
- La constante `c`, la forma del modelo y el año de inicio del entrenamiento se eligieron mirando la validación 2024 (permitido). **2025 no se ha tocado.**

In [ ]:
modelo_e1, cols_e1 = glm_poisson()
e1 = {}
for nombre, panel in paneles.items():
    datos_n = anadir_rezagos(panel, lags=(1, 12), ventanas=(3, 6, 12))
    e1[nombre] = evaluar_rolling(datos_n, MESES_VALIDACION, modelo_e1, cols_e1)
ENTRENO_DESDE

### E1 frente al listón (`media_12_estacional`)

In [ ]:
comparativa = {
    nombre: pd.DataFrame({
        "media_12_estacional": resumir(e0[nombre][0]["media_12_estacional"]),
        "E1 GLM Poisson": resumir(e1[nombre]),
    }).T.round(3)
    for nombre in paneles
}
comparativa["1km"]

In [ ]:
comparativa["2km"]

In [ ]:
comparativa["distritos"]

**Lectura:** E1 mejora claramente la **devianza de Poisson** a 1 km y 2 km (≈ 12 %) y empata en distritos; en **MAE y RMSE** queda igual o hasta un 1 % peor, y el **Capture@10 %** no cambia. Es lo esperable: E1 es esencialmente «media de 12 meses con un efecto de mes aprendido», y eso ya lo hacía el baseline estacional. El valor está en que da una distribución de probabilidad coherente (devianza) y sirve de base para E2.

### Errores por banda y por mes (1 km)

In [ ]:
pd.concat({
    "media_12_estacional": resumir_por_banda(e0["1km"][0]["media_12_estacional"]).set_index("banda"),
    "E1": resumir_por_banda(e1["1km"]).set_index("banda"),
}).round(3)

In [ ]:
pd.DataFrame({
    "media_12_estacional": resumir_por_mes(e0["1km"][0]["media_12_estacional"]).set_index("target_month")["MAE"],
    "E1": resumir_por_mes(e1["1km"]).set_index("target_month")["MAE"],
}).round(3).plot(marker="o", figsize=(7, 3), title="MAE por mes en 2024 (1 km)")

### Contraste de sobredispersión
Si la varianza fuese mucho mayor que la media, Poisson se quedaría corta y habría que pasar a Binomial Negativa. Contraste de Cameron y Trivedi: `alfa` en `Var = μ + alfa·μ²` (0 = Poisson exacta) y razón de Pearson (≈ 1 si Poisson ajusta bien).

In [ ]:
filas = {}
for nombre, panel in paneles.items():
    datos_n = anadir_rezagos(panel, lags=(1, 12), ventanas=(3, 6, 12))
    fuera = contraste_sobredispersion(e1[nombre]["y"], e1[nombre]["yhat"])
    filas[nombre] = {"alfa (2024)": fuera["alfa"], "p (2024)": fuera["p"],
                     "Pearson (2024)": fuera["pearson_chi2_por_n"]}
pd.DataFrame(filas).T.round(3)

**Lectura:** `alfa` entre 0,003 y 0,011 y razón de Pearson entre 0,95 y 1,45: hay algo de sobredispersión estadísticamente detectable en 2 km y distritos, pero **mínima en la práctica**. No justifica la Binomial Negativa; se documenta y se sigue con Poisson.

## Qué se probó con el GLM Poisson (para declararlo en la memoria)
La forma del modelo, la constante `c` y los años de entrenamiento se eligieron **mirando la validación 2024** (las directrices lo permiten; **2025 no se ha tocado**). Aun así, las cifras de validación pueden salir algo optimistas, y conviene decirlo.

**1. Forma del modelo** (pruebas exploratorias con modelo fijo: entreno hasta 2023, prueba 2024; 1 km; baseline: MAE 0,761 / devianza 0,640):

| Forma | MAE | Devianza |
|---|---|---|
| `log1p` de lag_1, lag_12, ma_3, ma_6, ma_12 + mes (L2 α = 0…0,1, protocolo mensual) | 1,034–1,036 | 0,954–0,969 |
| `log(x + 0,05)` de las 5 variables + mes | 0,803 | 0,612 |
| `log(x + 0,25)` de las 5 variables + mes | 0,869 | 0,715 |
| Offset `log(ma_12 + 0,05)` + mes | 0,803 | 0,606 |
| Offset `log(ma_12 + 0,25)` + mes | 0,914 | 0,803 |
| Offset `log(ma_12 + 0,001)` + mes | 0,777 | 0,581 |
| **Offset `log(ma_12 + 0,01)` + mes (elegido)** | 0,782 | 0,575 |
| Offset `log(ma_12 + 0,02)` + mes | 0,787 | 0,581 |
| Offset `log(ma_12 + 0,01)` + log-razones de lag_1, lag_12, ma_3, ma_6 + mes | 0,784 | 0,586 |

La `log1p` se descartó porque no era proporcional al historial. Una constante grande empeora porque da peso a celdas sin accidentes. Entre 0,001 y 0,02 casi no hay diferencia.

**2. Años de entrenamiento** (protocolo mensual completo, se puede reejecutar aquí):

In [ ]:
def sin_anios(f, anios):
    return lambda e, a: f(e[~e["target_month"].dt.year.isin(anios)], a)

glm_todo, _ = glm_poisson(entreno_desde=None)
variantes = {
    "con todo (desde 2019)": glm_todo,
    "sin 2020": sin_anios(glm_todo, [2020]),
    "desde 2022 (elegido)": modelo_e1,
}
sensibilidad = {}
for nombre, panel in paneles.items():
    datos_n = anadir_rezagos(panel, lags=(1, 12), ventanas=(3, 6, 12))
    ref = resumir(e0[nombre][0]["media_12_estacional"])
    sensibilidad[(nombre, "media_12_estacional (listón)")] = {
        "MAE": ref["MAE"], "devianza": ref["devianza_poisson"]}
    for etiqueta, f in variantes.items():
        r = resumir(evaluar_rolling(datos_n, MESES_VALIDACION, f, cols_e1))
        sensibilidad[(nombre, etiqueta)] = {"MAE": r["MAE"], "devianza": r["devianza_poisson"]}
pd.DataFrame(sensibilidad).T.round(3)

**Lectura:** a 1 km y 2 km la devianza de E1 es mejor que la del listón en las tres variantes y el MAE no mejora, así que la conclusión no depende del año de inicio. **En distritos sí depende:** con entreno desde 2019 la devianza de E1 es peor que la del baseline y solo con «desde 2022» llega al empate; ahí el resultado es frágil y se describe como que E1 no mejora al baseline.

**Sobre la justicia de la comparación:** todos usan solo información anterior al mes que predicen, y se evalúan sobre las mismas filas y métricas. Hay dos asimetrías que se compensan en parte: el índice estacional del baseline usa 2019 y 2021–2023 (más historia que E1 desde 2022), pero E1 tuvo más ajustes sobre la propia validación.